In [ ]:
def critic(answer, query, context):
    prompt = f"""
    Evaluate the answer based on the question and context.

    Question: {query}

    Context:
    {context}

    Answer:
    {answer}

    Is the answer correct and supported by the context?

    Reply only with:
    GOOD or BAD
    """

    response = llm.invoke(prompt).content.strip().lower()
    
    return "good" in response

In [ ]:
def refine_query(query, answer):
    prompt = f"""
    The previous answer was not good enough.

    Original Question: {query}
    Bad Answer: {answer}

    Improve the question to get a better answer:
    """

    response = llm.invoke(prompt).content.strip()
    return response

In [ ]:
def self_correcting_rag(query, max_iterations=3):
    
    print(f"\n🔍 Original Query: {query}")
    
    for i in range(max_iterations):
        print(f"\n🔁 Iteration {i+1}")
        
        # Retrieve
        docs = retriever.invoke(query)
        context = "\n".join([doc.page_content for doc in docs])
        
        # Generate
        prompt = f"""
        Answer the question using the context below.

        Context:
        {context}

        Question:
        {query}
        """
        
        answer = llm.invoke(prompt).content
        print(f"🧠 Answer: {answer}")
        
        # Critic check
        is_good = critic(answer, query, context)
        
        if is_good:
            print("✅ Answer is GOOD")
            return answer
        
        print("❌ Answer is BAD → refining...")
        
        # Improve query
        query = refine_query(query, answer)
        print(f"✏️ New Query: {query}")
    
    return answer

In [ ]:
query = "Explain RAG in detail"

final_answer = self_correcting_rag(query)

print("\n💡 Final Answer:\n", final_answer)